# field_check.ipynb — First-Light Frame Verification Tool

In-field go/no-go check: load a single plate-solved FITS frame taken at the
start of a variable-star imaging session, verify the target and 2-3 AAVSO
comparison stars are in-frame, away from the edges, and within the camera's
linear ADU range — before committing hours to the session.

**New dependencies** (not otherwise used in this repo):
```
pip install photutils opencv-python
```


In [1]:
# Cell 1 — Config

FITS_PATH = "sample_frames/Light/yzboofinal/Light_yzboofinal_90.0s_Bin1_20260808-205716_346deg_0002.fit"   # swap per session
TARGET_OBJECT_ID = "yzboo"                            # key into mobile/vs.json
VS_JSON_PATH = "mobile/vs.json"
COMP_STARS_DIR = "mobile/comp_stars"      # offline cache generated by vs_catalog.ipynb

SATURATION_ADU = 45000     # raw pixel ceiling (below full-well, with margin)
MIN_GOOD_ADU = 500         # minimum (peak - local background) for usable SNR
EDGE_MARGIN_PX = 150       # how close to a frame edge counts as "too close" (accounts for dither/drift)
CENTROID_SEARCH_PX = 15    # half-width of box used to refine each star's pixel position
PHOTOMETRY_RADIUS_PX = 5   # radius used to sample peak ADU / local background around each centroid
MIN_FWHM_PX = 2.0          # OSC cameras need a star's light spread across enough pixels for each
                           # 2x2 Bayer block to see a fair R/G/B mix; tighter stars undersample color
BAYER_PATTERN_FALLBACK = "RGGB"   # used only if the FITS header has no BAYERPAT keyword

STRETCH_A = 0.3            # AsinhStretch "a" param for the preview image (Cell 6) — higher = gentler/
                           # more linear, lower = more aggressive faint-detail boost (astropy default 0.1)


In [2]:
# Cell 2 — Imports

import json
import math
import warnings

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize
from astropy.modeling import models, fitting
from photutils.centroids import centroid_com


In [3]:
# Cell 3 — Load the FITS frame

with fits.open(FITS_PATH) as hdul:
    hdu = next(h for h in hdul if h.data is not None)
    raw_data = hdu.data.astype(np.float64)
    header = hdu.header

wcs = WCS(header)
frame_h, frame_w = raw_data.shape
bayer_pattern = header.get("BAYERPAT", BAYER_PATTERN_FALLBACK).strip()

# Plate scale sanity readout, derived from the WCS CD/PC matrix (arcsec/px)
pixel_scale_deg = np.sqrt(np.sum(wcs.pixel_scale_matrix ** 2, axis=0))
pixel_scale_arcsec = pixel_scale_deg * 3600.0

print(f"Frame: {frame_w}x{frame_h} px, dtype={hdu.data.dtype}, BAYERPAT={bayer_pattern}")
print(f"WCS plate scale: {pixel_scale_arcsec[0]:.3f} x {pixel_scale_arcsec[1]:.3f} arcsec/px")


FileNotFoundError: [Errno 2] No such file or directory: 'sample_frames/Light/yzboofinal/Light_yzboofinal_90.0s_Bin1_20260808-205716_346deg_0002.fit'

In [ ]:
# Cell 4 — Target + comparison star catalog for this frame (fully offline)

# Target's catalog RA/Dec from mobile/vs.json
vs_catalog = json.load(open(VS_JSON_PATH))
target_row = next(s for s in vs_catalog if s["objectId"] == TARGET_OBJECT_ID)
target_star = {
    "label": target_row["displayName"],
    "role": "target",
    "ra": target_row["ra"] * 15.0,   # vs.json stores RA in decimal hours
    "dec": target_row["dec"],
    "mag": target_row.get("magnitude"),
}

# Comparison stars from the offline cache generated by vs_catalog.ipynb — no network call.
comp_stars_path = f"{COMP_STARS_DIR}/{TARGET_OBJECT_ID}.json"
try:
    cached_comp_stars = json.load(open(comp_stars_path))
except FileNotFoundError:
    raise FileNotFoundError(
        f"No comparison-star cache at {comp_stars_path!r}. "
        f"Run vs_catalog.ipynb (with internet) to generate mobile/comp_stars/ first."
    )

comp_stars = [
    {
        "label": star["label"],   # AAVSO chart label, e.g. "89" == mag 8.9
        "role": "comp",
        "ra": star["ra"] * 15.0,   # cache stores RA in decimal hours, matching vs.json
        "dec": star["dec"],
        "mag": star["mag"],
    }
    for star in cached_comp_stars
]
stars = [target_star] + comp_stars

# Frame's true center RA/Dec + angular size, from the WCS (not the catalog value —
# actual pointing may differ slightly from nominal target coordinates) — diagnostic only.
center_ra, center_dec = (float(v) for v in wcs.pixel_to_world_values(frame_w / 2, frame_h / 2))
frame_diag_deg = np.hypot(frame_w * pixel_scale_deg[0], frame_h * pixel_scale_deg[1])

print(f"Frame center: RA={center_ra:.5f} Dec={center_dec:.5f}, FOV diag={frame_diag_deg:.3f} deg")
print(f"Loaded {len(comp_stars)} cached AAVSO comparison stars from {comp_stars_path}.")


In [ ]:
# Cell 5 — Locate, centroid, classify, and measure FWHM for each star

_gauss_fitter = fitting.LevMarLSQFitter()


def measure_fwhm(cutout, cx, cy):
    """Fit a 2D Gaussian to a background-subtracted cutout (star roughly at cx, cy in
    cutout coordinates) and return the FWHM in pixels, averaged across x/y. None on failure."""
    yy, xx = np.mgrid[0:cutout.shape[0], 0:cutout.shape[1]]
    amplitude = float(np.max(cutout))
    if amplitude <= 0:
        return None
    g_init = models.Gaussian2D(amplitude=amplitude, x_mean=cx, y_mean=cy,
                                x_stddev=2.0, y_stddev=2.0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        g_fit = _gauss_fitter(g_init, xx, yy, cutout)
    sigma = (abs(g_fit.x_stddev.value) + abs(g_fit.y_stddev.value)) / 2.0
    if not np.isfinite(sigma) or sigma <= 0:
        return None
    return sigma * 2.3548200450309493   # 2*sqrt(2*ln2)


def sample_star(raw, px, py):
    """Refine (px, py) to the local peak via a centroid, then sample raw peak ADU,
    local background, and FWHM around it.
    Returns (refined_px, refined_py, peak_adu, background_adu, fwhm_px)."""
    x0 = int(round(px)) - CENTROID_SEARCH_PX
    x1 = int(round(px)) + CENTROID_SEARCH_PX
    y0 = int(round(py)) - CENTROID_SEARCH_PX
    y1 = int(round(py)) + CENTROID_SEARCH_PX
    cutout = raw[y0:y1, x0:x1]
    if cutout.size == 0:
        return px, py, None, None, None

    local_bg = np.median(cutout)
    bg_sub_cutout = cutout - local_bg
    cx, cy = centroid_com(bg_sub_cutout)
    if not (np.isfinite(cx) and np.isfinite(cy)):
        refined_px, refined_py = px, py
        cx, cy = px - x0, py - y0
    else:
        refined_px, refined_py = x0 + cx, y0 + cy

    rx0 = int(round(refined_px)) - PHOTOMETRY_RADIUS_PX
    rx1 = int(round(refined_px)) + PHOTOMETRY_RADIUS_PX + 1
    ry0 = int(round(refined_py)) - PHOTOMETRY_RADIUS_PX
    ry1 = int(round(refined_py)) + PHOTOMETRY_RADIUS_PX + 1
    aperture = raw[ry0:ry1, rx0:rx1]
    peak_adu = float(np.max(aperture)) if aperture.size else None

    ann_r_out = PHOTOMETRY_RADIUS_PX * 3
    ax0, ax1 = int(round(refined_px)) - ann_r_out, int(round(refined_px)) + ann_r_out + 1
    ay0, ay1 = int(round(refined_py)) - ann_r_out, int(round(refined_py)) + ann_r_out + 1
    annulus = raw[ay0:ay1, ax0:ax1]
    background_adu = float(np.median(annulus)) if annulus.size else None

    # Saturated cores clip the Gaussian's peak and bias the fitted width low — skip the fit
    # rather than report a misleadingly tight FWHM for a star we already know is unusable.
    if peak_adu is not None and peak_adu <= SATURATION_ADU:
        fwhm_px = measure_fwhm(bg_sub_cutout, cx, cy)
    else:
        fwhm_px = None

    return refined_px, refined_py, peak_adu, background_adu, fwhm_px


results = []
for star in stars:
    coord = SkyCoord(ra=star["ra"], dec=star["dec"], unit="deg")
    px, py = wcs.world_to_pixel(coord)
    px, py = float(px), float(py)

    if not (0 <= px < frame_w and 0 <= py < frame_h):
        results.append({**star, "px": px, "py": py, "peak_adu": None,
                         "background_adu": None, "fwhm_px": None, "status": "OUT_OF_FRAME"})
        continue

    refined_px, refined_py, peak_adu, background_adu, fwhm_px = sample_star(raw_data, px, py)

    near_edge = (refined_px < EDGE_MARGIN_PX or refined_px > frame_w - EDGE_MARGIN_PX or
                 refined_py < EDGE_MARGIN_PX or refined_py > frame_h - EDGE_MARGIN_PX)

    if peak_adu is None:
        status = "OUT_OF_FRAME"
    elif peak_adu > SATURATION_ADU:
        status = "SATURATED"
    elif background_adu is not None and (peak_adu - background_adu) < MIN_GOOD_ADU:
        status = "TOO_FAINT"
    elif near_edge:
        status = "NEAR_EDGE"
    elif fwhm_px is not None and fwhm_px < MIN_FWHM_PX:
        status = "UNDERSAMPLED"
    else:
        status = "GOOD"

    results.append({**star, "px": refined_px, "py": refined_py, "peak_adu": peak_adu,
                     "background_adu": background_adu, "fwhm_px": fwhm_px, "status": status})

results_df = pd.DataFrame(results)[["label", "role", "px", "py", "peak_adu", "background_adu", "fwhm_px", "status"]]
print(results_df.to_string(index=False))


In [ ]:
# Cell 6 — Debayer + stretch for the preview

BAYER_CV_CODES = {
    "RGGB": cv2.COLOR_BAYER_RG2RGB,
    "BGGR": cv2.COLOR_BAYER_BG2RGB,
    "GRBG": cv2.COLOR_BAYER_GR2RGB,
    "GBRG": cv2.COLOR_BAYER_GB2RGB,
}
cv_code = BAYER_CV_CODES.get(bayer_pattern, BAYER_CV_CODES[BAYER_PATTERN_FALLBACK])

raw_u16 = np.clip(raw_data, 0, 65535).astype(np.uint16)
debayered = cv2.cvtColor(raw_u16, cv_code)   # RGB, for display only — raw_data (Cell 5) is untouched

luminance = debayered.mean(axis=2)
norm = ImageNormalize(luminance, interval=ZScaleInterval(), stretch=AsinhStretch(a=STRETCH_A))
stretched = norm(luminance)

preview_rgb = np.zeros_like(debayered, dtype=np.float64)
for c in range(3):
    channel = debayered[:, :, c].astype(np.float64)
    denom = np.maximum(luminance, 1e-6)
    preview_rgb[:, :, c] = np.clip(stretched * (channel / denom), 0, 1)


In [ ]:
# Cell 7 — Annotated plot

import matplotlib.patheffects as patheffects

STATUS_COLORS = {
    "GOOD": "#2E8B57",         # sea green (darkened from neon so it doesn't wash out against a halo)
    "NEAR_EDGE": "#FFEA00",    # bright yellow
    "UNDERSAMPLED": "#FF8C00", # dark orange
    "SATURATED": "#FF1744",    # bright red
    "TOO_FAINT": "#FF1744",    # bright red
    "OUT_OF_FRAME": "gray",
}
TARGET_COLOR = "#FF0000"   # always red, regardless of status — the target must stand out from comps at a glance
ANNOTATION_RADIUS_PX = PHOTOMETRY_RADIUS_PX * 5   # bigger circles read clearly at full-frame zoom
LABEL_FONTSIZE = 44   # same size for target and comps
# White halo, not black — the sky background is dark in nearly every frame, so a black outline
# disappears into it. White holds up against both dark sky and bright nebulosity/stars.
OUTLINE_EFFECT = [patheffects.withStroke(linewidth=4, foreground="white")]

dpi = 100
fig, ax = plt.subplots(figsize=(frame_w / dpi, frame_h / dpi), dpi=dpi)
ax.imshow(preview_rgb, origin="upper")
ax.axis("off")

for r in results:
    if r["status"] == "OUT_OF_FRAME":
        continue
    is_target = r["role"] == "target"
    color = TARGET_COLOR if is_target else STATUS_COLORS[r["status"]]
    circ = Circle((r["px"], r["py"]), radius=ANNOTATION_RADIUS_PX,
                   edgecolor=color, facecolor="none",
                   linewidth=4 if is_target else 2.5,
                   linestyle="--" if is_target else "-")
    circ.set_path_effects(OUTLINE_EFFECT)
    ax.add_patch(circ)
    # Just the star name/label here — peak ADU, FWHM, and status are still in the printed
    # results table (Cell 5); dropping them from the plot leaves room for much bigger text
    # without labels overlapping in crowded fields.
    label_text = f"★ {r['label']}" if is_target else r["label"]
    txt = ax.text(r["px"] + ANNOTATION_RADIUS_PX + 8, r["py"], label_text,
                   color=color, fontsize=LABEL_FONTSIZE,
                   fontweight="bold")
    txt.set_path_effects(OUTLINE_EFFECT)

plt.tight_layout()
plt.show()

n_comps = sum(1 for r in results if r["role"] == "comp")
n_comps_good = sum(1 for r in results if r["role"] == "comp" and r["status"] == "GOOD")
target_status = next(r["status"] for r in results if r["role"] == "target")
target_fwhm = next(r["fwhm_px"] for r in results if r["role"] == "target")
good_fwhms = [r["fwhm_px"] for r in results if r["fwhm_px"] is not None]
print(f"Target ({TARGET_OBJECT_ID}): {target_status}"
      + (f", FWHM={target_fwhm:.2f}px" if target_fwhm is not None else ""))
if good_fwhms:
    print(f"Median FWHM across measured stars: {np.median(good_fwhms):.2f}px "
          f"(min {MIN_FWHM_PX}px for this OSC sensor)")
print(f"Comparison stars usable: {n_comps_good}/{n_comps} GOOD "
      f"({', '.join(r['label'] + ':' + r['status'] for r in results if r['role'] == 'comp')})")
